# CBR with Ranked Mechanistic Path Extraction
Implements Case-Based Reasoning (Das et al. 2020) on MIND dataset.
Outputs:
- `predictions_test.tsv` — ranked drug-disease predictions with best path
- `paths_test.json` — full ranked mechanistic paths per prediction

**Before running:** Add your MIND dataset splits as input data.
Splits should be at `/kaggle/working/data/splits/slice_N/`
If splits don't exist, Cell 2 will create them from raw MIND files.


In [ ]:
# ── CELL 1: Imports and config ────────────────────────────────────────────
import pandas as pd
import numpy as np
import json
import random
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm

# ── Config ────────────────────────────────────────────────────────────────
INDICATION_REL = 'indication'   # relation name for drug->disease
N_SPLITS       = 3              # number of random splits
RANDOM_SEED    = 42
K_NEIGHBOURS   = 100            # number of similar drugs for CBR
MAX_PATH_LEN   = 3              # max hops for path extraction
MAX_BFS_PATHS  = 300            # max paths explored per drug
TOP_N_PATHS    = 5              # top N ranked paths saved per prediction

# Paths
SPLITS_DIR = Path('/kaggle/working/data/splits')
RESULTS    = Path('/kaggle/working/results/cbr/CBR')
RESULTS.mkdir(parents=True, exist_ok=True)

print('Config loaded')
print(f'Splits dir: {SPLITS_DIR}')
print(f'Results:    {RESULTS}')


In [ ]:
# ── CELL 2: Check/create splits ───────────────────────────────────────────
# If splits already exist from main notebook, skip this cell
# Otherwise, create them from raw MIND files

slices = sorted(SPLITS_DIR.glob('slice_*'))
if len(slices) >= N_SPLITS:
    print(f'Found {len(slices)} existing splits — using them')
    for sl in slices:
        files = [f.name for f in sl.iterdir()]
        print(f'  {sl.name}: {files}')
else:
    print('Splits not found — creating from raw MIND files...')
    
    # Find MIND files
    INPUT = Path('/kaggle/input')
    train_files = sorted(INPUT.glob('**/train.txt'), 
                         key=lambda f: f.stat().st_size, reverse=True)
    
    if not train_files:
        print('ERROR: No train.txt found in /kaggle/input')
        print('Add MIND dataset via Add Data button')
    else:
        BASE = train_files[0].parent
        print(f'Loading MIND from {BASE}...')
        
        train = pd.read_csv(BASE/'train.txt', sep='\t', header=None,
                            names=['head','relation','tail'])
        valid = pd.read_csv(BASE/'valid.txt', sep='\t', header=None,
                            names=['head','relation','tail'])
        test  = pd.read_csv(BASE/'test.txt',  sep='\t', header=None,
                            names=['head','relation','tail'])
        
        mrn = pd.concat([train, valid, test], ignore_index=True)
        print(f'Total triples: {len(mrn):,}')
        
        ind = mrn[mrn.relation == INDICATION_REL].reset_index(drop=True)
        print(f'Indication triples: {len(ind):,}')
        
        non_ind = mrn[mrn.relation != INDICATION_REL]
        nodes   = pd.Series(
            pd.unique(mrn[['head','tail']].values.ravel())
        ).dropna().unique()
        
        seeds = [RANDOM_SEED + i*100 for i in range(N_SPLITS)]
        
        for i, seed in enumerate(seeds):
            sl = SPLITS_DIR / f'slice_{i}'
            sl.mkdir(parents=True, exist_ok=True)
            
            idx = list(range(len(ind)))
            random.seed(seed)
            random.shuffle(idx)
            n      = len(idx)
            n_test = int(n * 0.10)
            n_val  = int(n * 0.10)
            
            tr = ind.iloc[idx[n_test+n_val:]]
            te = ind.iloc[idx[:n_test]]
            va = ind.iloc[idx[n_test:n_test+n_val]]
            
            kge_train = pd.concat([non_ind, tr], ignore_index=True)
            kge_train.to_csv(sl/'kge_train.tsv', sep='\t',
                             index=False, header=False)
            tr.to_csv(sl/'ind_train.tsv', sep='\t', index=False, header=False)
            te.to_csv(sl/'ind_test.tsv',  sep='\t', index=False, header=False)
            va.to_csv(sl/'ind_valid.tsv', sep='\t', index=False, header=False)
            pd.Series(nodes).to_csv(sl/'entities.txt',
                                    index=False, header=False)
            print(f'  slice_{i}: train={len(tr)} test={len(te)} valid={len(va)}')
        
        print('Splits created!')


In [ ]:
# ── CELL 3: Graph builder ─────────────────────────────────────────────────

def build_graph(tsv_path):
    """
    Build adjacency list from full KGE training graph.
    Includes inverse edges for bidirectional traversal.
    graph[entity] = [(relation, neighbour), ...]
    """
    print(f'  Building graph from {Path(tsv_path).name}...')
    df = pd.read_csv(tsv_path, sep='\t', header=None,
                     names=['h','r','t'])
    graph = defaultdict(list)
    for _, row in tqdm(df.iterrows(), total=len(df),
                       desc='  Loading edges', leave=False):
        graph[row.h].append((row.r, row.t))
        graph[row.t].append((f'inv_{row.r}', row.h))
    print(f'  Graph: {len(graph):,} nodes, {len(df)*2:,} directed edges')
    return dict(graph)

print('Graph builder ready')


In [ ]:
# ── CELL 4: Relation quality weights ──────────────────────────────────────
# Used to rank paths — higher weight = more mechanistically meaningful

RELATION_WEIGHTS = {
    # Direct mechanistic — highest quality
    'inhibits':             1.0,
    'activates':            1.0,
    'marker_or_mechanism':  1.0,
    'treats':               1.0,
    'causes':               0.9,
    'prevents':             0.9,
    'palliates':            0.8,
    'disrupts':             0.8,
    # Indirect mechanistic — medium quality
    'affects':              0.7,
    'regulates':            0.7,
    'positively_regulates': 0.7,
    'negatively_regulates': 0.7,
    'produces':             0.6,
    'capable_of':           0.6,
    # Structural / taxonomic — lower quality
    'associated_with':      0.4,
    'presents':             0.4,
    'site_of':              0.3,
    'part_of':              0.3,
    'in_reaction_with':     0.3,
    'in_taxon':             0.1,
}

def get_rel_weight(rel):
    """Get quality weight — strip MRN suffix codes like _CinG."""
    clean = rel.replace('inv_', '')
    # Try progressively shorter prefixes
    parts = clean.split('_')
    for n_parts in range(len(parts), 0, -1):
        candidate = '_'.join(parts[:n_parts])
        if candidate in RELATION_WEIGHTS:
            return RELATION_WEIGHTS[candidate]
    return 0.2  # default for unknown relations

# Test
test_rels = ['inhibits_CinG', 'marker_or_mechanism_GmD',
             'associated_with_BPawD', 'inv_treats_CtD']
print('Relation weight test:')
for r in test_rels:
    print(f'  {r:45s} -> {get_rel_weight(r)}')


In [ ]:
# ── CELL 5: Path finding functions ────────────────────────────────────────

def format_path(drug, path_steps):
    """Format path as readable string."""
    if not path_steps:
        return ''
    parts = [str(drug)]
    for rel, entity in path_steps:
        rel_display = rel.replace('inv_', '← ').replace('_', ' ')
        parts.append(f'--[{rel_display}]-->')
        parts.append(str(entity))
    return ' '.join(parts)


def find_ranked_paths(drug, target_disease, graph,
                      max_len=3, max_bfs=300):
    """
    BFS from drug to target_disease, collecting all paths up to max_len hops.
    
    Ranking score = geometric_mean(relation_weights) / path_length
    Higher score = shorter path with higher quality relations.
    
    Returns list of dicts sorted by score descending:
      {'score', 'length', 'path', 'relations'}
    """
    # BFS queue: (current_entity, path_steps, cumulative_quality)
    queue   = [(drug, [], 1.0)]
    visited = {drug}
    found   = []       # paths that reach target_disease
    n_explored = 0

    while queue and n_explored < max_bfs:
        curr, path, cum_q = queue.pop(0)

        if len(path) >= max_len:
            continue

        for rel, nb in graph.get(curr, []):
            rel_w   = get_rel_weight(rel)
            new_path = path + [(rel, nb)]
            new_q    = cum_q * rel_w
            n_explored += 1

            # Prune: skip very low quality at depth 1
            if len(path) == 0 and rel_w < 0.3:
                continue

            if nb == target_disease:
                # Reached target — compute score
                path_len  = len(new_path)
                mean_qual = new_q ** (1.0 / path_len)
                score     = mean_qual / path_len
                found.append({
                    'score':     round(float(score), 4),
                    'length':    path_len,
                    'path':      format_path(drug, new_path),
                    'relations': ' -> '.join(r for r,e in new_path),
                })

            # Continue BFS
            if len(new_path) < max_len and nb not in visited:
                visited.add(nb)
                queue.append((nb, new_path, new_q))

    # Sort by score, deduplicate by path string
    found.sort(key=lambda x: -x['score'])
    seen, ranked = set(), []
    for p in found:
        if p['path'] not in seen:
            seen.add(p['path'])
            ranked.append(p)

    return ranked


print('Path finding functions ready')


In [ ]:
# ── CELL 6: CBR scoring function ──────────────────────────────────────────

def cbr_score(drug, graph, tr_triples, k, relation):
    """
    CBR: score candidate diseases for a query drug.
    
    Algorithm:
    1. Find k most similar drugs (overlap in outgoing edge types)
    2. Score diseases by weighted vote from similar drugs' known indications
    
    Returns dict: {disease: score}
    """
    # Step 1: find similar drugs via shared graph edges
    query_edges = set((r, t) for r, t in graph.get(drug, []))
    
    nb_scores = {}
    for h, r, t in tr_triples:
        if h == drug:
            continue
        nb_edges = set((r2, t2) for r2, t2 in graph.get(h, []))
        overlap  = len(query_edges & nb_edges)
        if overlap > 0:
            nb_scores[h] = nb_scores.get(h, 0) + overlap

    top_nb = sorted(nb_scores, key=nb_scores.get, reverse=True)[:k]
    total  = sum(nb_scores.get(n, 1) for n in top_nb) or 1

    # Step 2: weighted vote for diseases
    disease_scores = defaultdict(float)
    for nb in top_nb:
        w = nb_scores.get(nb, 1) / total
        for h, r, t in tr_triples:
            if h == nb and r == relation:
                disease_scores[t] += w

    return dict(disease_scores)


print('CBR scoring function ready')


In [ ]:
# ── CELL 7: Main CBR run with path extraction ─────────────────────────────
# Runs CBR on all slices, extracts and ranks mechanistic paths

slices = sorted(SPLITS_DIR.glob('slice_*'))
print(f'Running CBR on {len(slices)} slices with max_path_len={MAX_PATH_LEN}\n')

for sl in slices:
    outdir = RESULTS / sl.name
    outdir.mkdir(parents=True, exist_ok=True)

    # Build graph once per slice (reused for both test and valid)
    graph = build_graph(sl / 'kge_train.tsv')

    # Load training indication triples
    tr_df      = pd.read_csv(sl/'ind_train.tsv', sep='\t',
                             header=None, names=['h','r','t'])
    tr_triples = list(tr_df.itertuples(index=False, name=None))
    all_ents   = pd.read_csv(sl/'entities.txt', header=None)[0].tolist()
    n_ents     = len(all_ents)

    print(f'=== {sl.name} ===')
    print(f'  Training indication triples: {len(tr_triples):,}')
    print(f'  Total entities: {n_ents:,}')

    for split in ['test', 'valid']:
        pred_out  = outdir / f'predictions_{split}.tsv'
        paths_out = outdir / f'paths_{split}.json'

        if pred_out.exists() and paths_out.exists():
            df_existing = pd.read_csv(pred_out, sep='\t')
            print(f'  SKIP {split} — already done '
                  f'(MRR={df_existing.reciprocal_rank.mean():.4f})')
            continue

        print(f'\n  Running {split} split...')
        ev_df    = pd.read_csv(sl/f'ind_{split}.tsv', sep='\t',
                               header=None, names=['h','r','t'])
        ev_queries = list(ev_df.itertuples(index=False, name=None))
        print(f'  Queries: {len(ev_queries):,}')

        rows      = []
        all_paths = {}  # drug -> {disease -> [ranked_paths]}

        for drug, rel, expected in tqdm(ev_queries,
                                        desc=f'  CBR {split}'):
            # Step 1: CBR scoring
            scores = cbr_score(drug, graph, tr_triples,
                               K_NEIGHBOURS, INDICATION_REL)

            if not scores:
                rank     = n_ents
                sd       = []
                top_pred = ''
            else:
                sd       = sorted(scores, key=scores.get, reverse=True)
                rank     = sd.index(expected) + 1 if expected in sd else n_ents
                top_pred = sd[0]

            # Step 2: Path extraction for expected disease + top 5 predictions
            targets = list(dict.fromkeys(
                ([expected] if expected in scores else []) + sd[:5]
            ))

            drug_paths = {}
            for target in targets:
                ranked = find_ranked_paths(
                    drug, target, graph,
                    max_len=MAX_PATH_LEN,
                    max_bfs=MAX_BFS_PATHS
                )
                if ranked:
                    drug_paths[target] = ranked[:TOP_N_PATHS]

            all_paths[drug] = drug_paths

            # Best path to expected disease
            best_path  = ''
            path_score = 0.0
            n_paths    = 0
            if expected in drug_paths and drug_paths[expected]:
                best_path  = drug_paths[expected][0]['path']
                path_score = drug_paths[expected][0]['score']
                n_paths    = len(drug_paths[expected])

            rows.append({
                'drug':             drug,
                'expected_disease': expected,
                'rank':             rank,
                'reciprocal_rank':  round(1.0 / rank, 6),
                'top_prediction':   top_pred,
                'best_path':        best_path,
                'best_path_score':  path_score,
                'n_paths_found':    n_paths,
            })

        # Save predictions TSV
        df_out = pd.DataFrame(rows)
        df_out.to_csv(pred_out, sep='\t', index=False)

        # Save full ranked paths JSON
        with open(paths_out, 'w') as f:
            json.dump(all_paths, f, indent=2)

        # Report
        mrr        = df_out.reciprocal_rank.mean()
        hits1      = (df_out.rank == 1).mean()
        hits3      = (df_out.rank <= 3).mean()
        hits10     = (df_out.rank <= 10).mean()
        with_paths = (df_out.n_paths_found > 0).sum()

        print(f'\n  Results ({split}):')
        print(f'    MRR:        {mrr:.4f}')
        print(f'    Hits@1:     {hits1:.4f}')
        print(f'    Hits@3:     {hits3:.4f}')
        print(f'    Hits@10:    {hits10:.4f}')
        print(f'    With paths: {with_paths}/{len(df_out)}')
        print(f'    Saved:      {pred_out.name}, {paths_out.name}')

print('\n=== CBR complete! ===')


In [ ]:
# ── CELL 8: Show example ranked paths ────────────────────────────────────
# Display the most interesting mechanistic paths found

import json
from pathlib import Path

print('=== Example Mechanistic Paths ===')
print('(Showing cases where expected disease was found with paths)\n')

shown = 0
for sl in sorted(RESULTS.glob('slice_*')):
    paths_file = sl / 'paths_test.json'
    preds_file = sl / 'predictions_test.tsv'
    if not paths_file.exists():
        continue

    with open(paths_file) as f:
        all_paths = json.load(f)

    df = pd.read_csv(preds_file, sep='\t')

    # Find cases with paths to expected disease, ranked in top 10
    good = df[(df.n_paths_found > 0) & (df.rank <= 10)].head(5)

    for _, row in good.iterrows():
        drug_paths = all_paths.get(row.drug, {})
        exp_paths  = drug_paths.get(row.expected_disease, [])
        if not exp_paths:
            continue

        print(f'Drug:     {row.drug}')
        print(f'Disease:  {row.expected_disease}')
        print(f'Rank:     {int(row.rank)}')
        print(f'Paths to disease ({len(exp_paths)} found):')
        for j, p in enumerate(exp_paths[:3]):
            print(f'  [{j+1}] score={p["score"]:.3f} '
                  f'length={p["length"]} hops')
            print(f'      {p["path"]}')
        print()
        shown += 1
        if shown >= 10:
            break
    if shown >= 10:
        break

if shown == 0:
    print('No paths found yet — run Cell 7 first')


In [ ]:
# ── CELL 9: Summary table across all slices ───────────────────────────────

print('=== CBR Results Summary ===')
print(f'{"Slice":<10} {"Split":<8} {"MRR":<8} {"H@1":<8} {"H@3":<8} {"H@10":<8} {"With paths"}')
print('-' * 65)

for sl in sorted(RESULTS.glob('slice_*')):
    for split in ['test', 'valid']:
        f = sl / f'predictions_{split}.tsv'
        if not f.exists():
            continue
        df = pd.read_csv(f, sep='\t')
        mrr   = df.reciprocal_rank.mean()
        hits1 = (df.rank == 1).mean()
        hits3 = (df.rank <= 3).mean()
        hits10= (df.rank <= 10).mean()
        wp    = (df.n_paths_found > 0).sum()
        print(f'{sl.name:<10} {split:<8} {mrr:<8.4f} {hits1:<8.4f} '
              f'{hits3:<8.4f} {hits10:<8.4f} {wp}/{len(df)}')

print('\nFiles saved:')
for f in sorted(RESULTS.glob('**/*.tsv')):
    print(f'  {f}')
for f in sorted(RESULTS.glob('**/*.json')):
    size = f.stat().st_size / 1024
    print(f'  {f}  ({size:.0f} KB)')


In [ ]:
# ── CELL 10: Package results for download ────────────────────────────────
import shutil

shutil.make_archive('/kaggle/working/cbr_results', 'zip',
                    root_dir='/kaggle/working/results/cbr')
size = Path('/kaggle/working/cbr_results.zip').stat().st_size / 1024**2
print(f'Packaged: cbr_results.zip ({size:.1f} MB)')
print('Download via: Output tab -> cbr_results.zip')
print()
print('Contents:')
print('  CBR/slice_N/predictions_test.tsv  — ranked predictions')
print('  CBR/slice_N/paths_test.json       — ranked mechanistic paths')
print('  CBR/slice_N/predictions_valid.tsv — validation predictions')
print('  CBR/slice_N/paths_valid.json      — validation paths')
